# Lekcia 18: Zabezpečenie AI agentov pomocou kryptografických príjmov

## Praktický zošit

Tento zošit vás prevedie štyrmi cvičeniami:

1. **Podpíšte svoj prvý príjem** pre volanie nástroja agenta a overte ho.
2. **Pozmeňte príjem** a sledujte, ako overenie zlyhá.
3. **Vytvorte reťaz trojich príjmov** a potvrďte integritu reťaze.
4. **Zabaliť volanie nástroja Microsoft Agent Framework** tak, aby každá akcia vyprodukovala príjem.

Všetky kryptografické primitívy sú importované z dobre udržiavaných knižníc (`pynacl` pre Ed25519, `jcs` pre RFC 8785 kanonický JSON, `hashlib` zo štandardnej Python knižnice pre SHA-256). Logika príjmov samotná je obyčajný Python, ktorý môžete čítať a upravovať.

Spúšťajte bunky v poradí. Každá sekcia je krátka a samostatná.


## Nastavenie

Nainštalujte dve závislosti. Obe majú permisívne licencie (Apache-2.0 / MIT).


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## Pomocné nástroje

Tieto dva pomocné nástroje sa zaoberajú kódovaním base64url (bez doplňovania) a hashovaním SHA-256 ľubovoľných objektov. Zvyšok poznámkového bloku tak môže zostať zameraný priamo na logiku príjmu.


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Časť 1: Podpíšte svoj prvý doklad

Predstavte si, že náš agent pre **Contoso Travel** práve vyhľadal lety zo Sydney do Los Angeles pre zákazníka. Chceme zaznamenať tento hovor s nástrojom ako podpísaný doklad, aby si ho budúci kontrolór mohol overiť bez toho, aby nám musel veriť.

### Krok 1.1: Vygenerujte podpisový kľúč

V produkcii by podpisový kľúč agenta bol uložený v hardvérovom bezpečnostnom module (HSM), Azure Key Vault alebo v podobnom chránenom úložisku. Pre túto lekciu vygenerujeme nový kľúč v pamäti.


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### Krok 1.2: Vytvorenie obsahu potvrdenia

Obsah obsahuje všetko, čomu chceme, aby potvrdenie potvrdilo: kto konal, aký nástroj, s akými argumentmi, čo sa vrátilo, podľa akej politiky a kedy. Argumenty a výsledok hashuje, namiesto aby ich zahrnul priamo, aby potvrdenie neprepúšťalo citlivý obsah.


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### Krok 1.3: Podpísať a zostaviť potvrdenie

Tri kroky:

1. Kanonikalizujte obsah pomocou JCS tak, aby dve implementácie produkujúce rovnaké logické potvrdenie vytvorili bajtovo totožné bajty.
2. Zahashujte kanonické bajty pomocou SHA-256.
3. Podpíšte hash súkromným kľúčom Ed25519.

Podpis sa potom pripojí k pôvodnému obsahu, aby vzniklo konečné potvrdenie.


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### Krok 1.4: Overenie potvrdenia

Overenie proces obráti. Odstránime podpis, znovu vypočítame kanonický hash a skontrolujeme podpis voči verejnému kľúču v potvrdení.

Auditor vykonávajúci toto overenie nepotrebuje nič okrem samotného potvrdenia. Nemusí volať žiadnu službu, dotazovať sa v adresári kľúčov ani vyžadovať dôveru.


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

Mali by ste vidieť `Príjmový doklad je platný: True`. Agent vytvoril svoj prvý kryptograficky podpísaný auditný záznam.


## Sekcia 2: Zneužiť potvrdenie

Celý zmysel potvrdení je, že sú odolné voči manipulácii. Poďme to dokázať.

Zmeníme presne jeden znak na potvrdení a pozrieme sa, ako overenie zlyhá.


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### Čo sa práve stalo?

Keď sme zmenili `policy_id`, zmenili sa kanonické bajty. SHA-256 hash týchto bajtov sa zmenil. Podpis (ktorý bol nad pôvodným hashom) už nezodpovedá novému hashu. Overenie správne vráti `False`.

Neexistuje spôsob, ako upraviť akékoľvek pole v doklade a pritom ho stále overiť, pokiaľ útočník nemá súkromný kľúč. Pokiaľ je súkromný kľúč uložený v trezore kľúčov a verejný kľúč je publikovaný, manipuláciu nie je možné skryť.

Vyskúšajte to sami: upravte `tool_name` alebo `agent_id` alebo `timestamp` v bunke vyššie a spustite znova. Každá zmena vytvorí neplatný doklad.


## Sekcia 3: Spájanie potvrdení do reťazca

Jedno potvrdenie chráni jednu akciu. Väčšina agentov vykoná mnoho akcií. Aby bol celý sled nezmeniteľný, pripojíme každé potvrdenie k predchádzajúcemu tým, že do nákladu nového potvrdenia zahrnieme hash predchádzajúceho potvrdenia.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

Ak niekto odstráni alebo preusporiada potvrdenie, reťaz sa zlomí práve v tom bode. Overenie akéhokoľvek neskoršieho potvrdenia zlyhá, pretože jeho `previous_receipt_hash` už nezodpovedá skutočnému hashu jeho predchodcu.


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Teraz prerušte reťazec manipuláciou so stredným potvrdením a overte ho znova. Manipulované potvrdenie neprejde kontrolou podpisu, A ďalšie potvrdenie neprejde kontrolou reťazového odkazu (pretože jeho `previous_receipt_hash` už nezodpovedá hašu upraveného stredného potvrdenia).


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Potvrdenie 0 sa stále overuje (nebolo upravené a nemá predchodcu, na ktorom by záviselo). Potvrdenie 1 neprejde overením podpisu, pretože sme zmenili `tool_args_hash`. Potvrdenie 2 neprejde overením reťazového odkazu, pretože jeho `previous_receipt_hash` bol vypočítaný z pôvodného (teraz upraveného) potvrdenia 1.

Aj keby útočník znovu podpísal upravené potvrdenie 1 (čo nemôže robiť bez súkromného kľúča), nezrovnalosť v reťazovom odkaze potvrdenia 2 by stále odhalila manipuláciu. Na ukrytie zmeny by útočník musel znovu podpísať každé potvrdenie od miesta úpravy ďalej, čo vyžaduje vlastníctvo súkromného kľúča.


## Sekcia 4: Zabaľte volanie nástroja agenta s podpísaním potvrdenia

Pri reálnom nasadení nechcete, aby si každý autor agenta pamätal volať `make_receipt`. Chcete, aby podpisovanie potvrdenia prebiehalo automaticky pri každom vyvolaní nástroja.

Tu je najjednoduchší vzor: obalová trieda, ktorá prijme akúkoľvek volateľnú funkciu nástroja a vráti jej verziu generujúcu potvrdenie. Toto sa prispôsobí akémukoľvek rámcu agenta, vrátane Microsoft Agent Framework (`agent_framework.foundry`).

Ak nemáte nastavený projekt Microsoft Foundry, lokálny simulovaný príklad nižšie stále demonštruje tento vzor.


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


### Integrácia s Microsoft Agent Framework

Zabalič `ReceiptedTool` vyššie je nezávislý od frameworku. Ak ho chcete použiť v agente vytvorenom pomocou Microsoft Agent Framework, zaregistrujte zabalenú funkciu ako nástroj. Náčrt (nahradili by ste mock reálnou registráciou nástroja Microsoft Foundry):

```python
# Pseudokód zobrazujúci tvar integrácie.
# import os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="Ste agent Contoso Travel ...",
#     tools=[receipted_lookup],   # zabalený nástroj, nie surová funkcia
# )
# response = agent.run("Nájdite lety zo Sydney do Los Angeles v júni.")
#
# # Po spustení má každý volanie nástroja agenta podpísaný potvrdenie:
# audit_chain = receipted_lookup.receipts
```

Agentný framework nemusí nič vedieť o potvrdeniach. Podpisovanie potvrdení je zabalené okolo nástroja, nie pevne zabudované do frameworku. Takto pridáte pôvodnosť k existujúcemu kódu agenta bez prepísania agenta.


## Zhrnutie a náročná výzva

Dosiahli ste:

- Vygenerovali ste pár kľúčov Ed25519.
- Vytvorili a podpísali potvrdenku pre volanie nástroja agenta.
- Overili ste potvrdenku offline iba pomocou verejného kľúča.
- Manipulovali ste s potvrdenkou a pozorovali zlyhanie overenia.
- Vytvorili ste hashovo reťazenú sériu troch potvrdeniek.
- Zmenili ste prostredok reťazca a pozorovali ste zlyhanie podpisu i reťazového odkazu.
- Zabalenie funkcie nástroja s automatickým podpisovaním potvrdeniek.

**Náročná výzva.** Rozšírte schému potvrdeniek o pole `request_id` (UUID pre distribuované trasovanie). Aktualizujte `make_receipt`, aby ho zahŕňal, a potvrďte, že potvrdenky sa stále overujú kompletne. Potom pole zmeňte po podpise a potvrďte, že overenie zlyhá. Toto vás núti pochopiť, ako každý bajt kanonického kódovania prispieva k podpisu.

**Dôležitá hranica.** Potvrdenky dokážu tri veci a iba tri veci: atribúciu (tento kľúč podpísal tento obsah), integritu (obsah sa od podpisu nezmenil) a poradie (táto potvrdenka prišla po tamtej potvrdenke). Nedokážu však dokázať, že akcia agenta bola správna, že politika uvedená v `policy_id` bola skutočne vyhodnotená alebo že agent dodržal všetky pravidlá. Potvrdenky sú základ. Správa je systém, ktorý na ňom stavia.

Prečítajte si znova README lekcie s touto hranicou na pamäti. Najbežnejšou chybou tímov pri potvrdenkách je predpoklad, že „máme potvrdenky“ znamená „sme spravovaní“. Nie je to tak. Potvrdenky umožňujú auditovať správanie agenta. Nerobia ho správnym.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vyhlásenie o zodpovednosti**:
Tento dokument bol preložený pomocou AI prekladateľskej služby [Co-op Translator](https://github.com/Azure/co-op-translator). Hoci sa snažíme o presnosť, vezmite prosím na vedomie, že automatické preklady môžu obsahovať chyby alebo nepresnosti. Pôvodný dokument v jeho natívnom jazyku by mal byť považovaný za autoritatívny zdroj. Pre kritické informácie sa odporúča profesionálny ľudský preklad. Nie sme zodpovední za žiadne nedorozumenia alebo nesprávne interpretácie vyplývajúce z použitia tohto prekladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
